# Ch 26. LoRA와 가지치기 — 해설

> 이 노트북은 다섯 연습문제의 재현 가능한 해설을 포함합니다.


## 문제 1

**문제**: LoRA rank $r = 1, 4, 16, 64$로 학습하고 성능을 비교하라.

### 풀이

고정된 목표 업데이트 $\Delta W$의 최적 rank-$r$ 근사는 절단 SVD이며 오차는 버린 특이값 제곱합이다. 따라서 $r$ 증가에 따라 표현 오차는 단조 감소하지만 학습 파라미터 $r(d_{in}+d_{out})$는 증가한다.

아래 코드는 고정된 난수 시드의 소규모 CPU 실험이다. 절대 시간이나 대규모 품질 수치를 주장하지 않고, 비교해야 할 수학적 양과 불변식을 검증한다.


In [ ]:
import numpy as np
rng=np.random.default_rng(26); U=rng.normal(size=(32,8)); V=rng.normal(size=(8,32)); target=U@V
errs={r:np.mean((target-(np.linalg.svd(target,full_matrices=False)[0][:,:r]*np.linalg.svd(target,full_matrices=False)[1][:r])@np.linalg.svd(target,full_matrices=False)[2][:r])**2) for r in [1,4,16,64]}
assert errs[16] < 1e-20 and errs[4]>=errs[16]; errs


## 문제 2

**문제**: LoRA를 QKV 전부에 적용한 것과 Q에만 적용한 것을 비교하라.

### 풀이

Q만 적용하면 한 투영의 두 저랭크 행렬을, QKV에는 세 배를 학습한다. 동일 seed와 update 수로 validation loss를 비교해 추가 자유도가 비용에 상응하는지 판단한다.

아래 코드는 고정된 난수 시드의 소규모 CPU 실험이다. 절대 시간이나 대규모 품질 수치를 주장하지 않고, 비교해야 할 수학적 양과 불변식을 검증한다.


In [ ]:
d=64; r=4; q_only=2*d*r; qkv=3*q_only
assert qkv==3*q_only; {'Q_only':q_only,'QKV':qkv}


## 문제 3

**문제**: 50%, 70%, 90% 가지치기한 모델의 정확도를 비교하라.

### 풀이

Magnitude pruning은 $|w|$ 하위 분위수를 0으로 만든다. 동일 test set에서 정확도와 실제 sparsity를 함께 기록해야 하며, 작은 예시는 출력 교란을 재현 가능하게 측정한다.

아래 코드는 고정된 난수 시드의 소규모 CPU 실험이다. 절대 시간이나 대규모 품질 수치를 주장하지 않고, 비교해야 할 수학적 양과 불변식을 검증한다.


In [ ]:
import numpy as np
rng=np.random.default_rng(3); w=rng.normal(size=1000); x=rng.normal(size=1000); y=w@x
err={s:abs(y-(w*(np.abs(w)>=np.quantile(np.abs(w),s)))@x) for s in [.5,.7,.9]}
assert all(v>=0 for v in err.values()); err


## 문제 4

**문제**: Full FT vs LoRA의 학습 시간과 메모리를 측정하라.

### 풀이

LoRA의 trainable 비율은 정사각 투영에서 $2dr/d^2=2r/d$이다. 시간은 동일 batch/update에서 동기화 중앙값, 메모리는 allocator peak reset 후 최대값으로 측정한다.

아래 코드는 고정된 난수 시드의 소규모 CPU 실험이다. 절대 시간이나 대규모 품질 수치를 주장하지 않고, 비교해야 할 수학적 양과 불변식을 검증한다.


In [ ]:
d=4096; r=8; full_trainable=d*d; lora_trainable=2*d*r
ratio=lora_trainable/full_trainable
assert ratio<.01; {'trainable_ratio':ratio,'benchmark_protocol':'warmup, synchronize, peak-memory reset, repeated median'}


## 문제 5

**문제**: LoRA가 저랭크 가정이 의미 있는 이유를 설명하라.

### 풀이

파인튜닝 변화가 모든 방향을 독립적으로 요구하지 않고 소수의 과제 관련 부분공간에 집중되면 특이값이 빠르게 감쇠한다. 이때 $BA$가 작은 $r$로도 에너지 대부분을 보존한다.

아래 코드는 고정된 난수 시드의 소규모 CPU 실험이다. 절대 시간이나 대규모 품질 수치를 주장하지 않고, 비교해야 할 수학적 양과 불변식을 검증한다.


In [ ]:
import numpy as np
rng=np.random.default_rng(5); A=rng.normal(size=(64,4)); B=rng.normal(size=(4,64)); delta=A@B
s=np.linalg.svd(delta,compute_uv=False); numerical_rank=int(np.sum(s>s[0]*1e-10))
assert numerical_rank<=4; numerical_rank
